# 3.1.1. Pure Spatio-Temporal Resolution Baseline Evaluation

This notebook evaluates the performance of 12 distinct spatio-temporal datasets to determine which combination of spatial boundaries and time windows provides the best balance of predictive power and stability for our taxi demand model.

To make this evaluation completely rigorous and fair, we are implementing a strict **feature isolation protocol**:
* **Explicit Spatial Targeting:** We look specifically for the exact boundary definition of each dataset (`pickup_h3_res6`, `pickup_h3_res7`, or `pickup_community_area`).
* **Zero Coordinate Leakage:** We strictly exclude continuous coordinate columns (`latitude` and `longitude`). If left in, the model would bypass the spatial boundary definitions entirely, invalidating the resolution comparison.
* **Categorical Encoding:** Spatial identifiers (even integer-based ones like community areas) are routed through a `OneHotEncoder` to prevent the model from assuming ordinal relationships (e.g., treating Area 46 as "greater than" Area 2).

---

#### Evaluation Pipeline Overview

1. **Chronological Sorting & Split:** Sorts data by `time_bucket` and splits it (80% train / 20% test) to prevent future data leakage.
2. **Sparsity Audit:** Computes the percentage of zero-demand periods to detect where resolutions become too granular to provide a reliable signal.
3. **Sparse Matrix Preprocessing:** Standard-scales continuous features and one-hot encodes the targeted spatial column into a highly memory-efficient sparse matrix.
4. **Primal Optimization Baseline:** Fits a `LinearSVR` model using `squared_epsilon_insensitive` loss, allowing for rapid $O(N)$ training across all 675,000 rows.

#### Key Decision Metrics

* **Zero Demand (%):** A metric above ~40-50% indicates high sparsity, where the resolution may be too granular, causing the model to struggle with over-predicting zeros.
* **NRMSE (%):** Normalized Root Mean Squared Error. This scales the error as a percentage of the dataset's average trip count, enabling an "apples-to-apples" error comparison across different time buckets. Lower percentages indicate higher reliability.
* **R-Squared ($R^2$):** Quantifies how much variance the model captures. A negative or near-zero $R^2$ serves as an immediate red flag that the resolution contains too much noise for a linear estimator.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import LinearSVR
from sklearn.metrics import r2_score, mean_squared_error
import pandas as pd
import numpy as np
import os
from pathlib import Path

In [2]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Reset working directory                 #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import __main__
_nb = getattr(__main__, "__vsc_ipynb_file__", None) or os.environ.get("JPY_SESSION_NAME")
_start = Path(_nb).resolve().parent if _nb else Path.cwd()
os.chdir(next(p for p in [_start, *_start.parents] if (p / "pyproject.toml").exists()))
print(f"Working directory: {os.getcwd()}")

Working directory: C:\Users\Yannick Herrmann\Documents\Uni\AAA\AAA_Project\AAA_TA_2026


In [7]:
# List of your dataset files
files = [
    "data/aggregated/community_area/demand_ca_1h.parquet", 
    "data/aggregated/community_area/demand_ca_2h.parquet", 
    "data/aggregated/community_area/demand_ca_6h.parquet", 
    "data/aggregated/community_area/demand_ca_24h.parquet",
    
    "data/aggregated/hexagon/demand_hex_1h_low.parquet", 
    "data/aggregated/hexagon/demand_hex_1h_medium.parquet", 
    "data/aggregated/hexagon/demand_hex_2h_low.parquet", 
    "data/aggregated/hexagon/demand_hex_2h_medium.parquet",
    "data/aggregated/hexagon/demand_hex_6h_low.parquet", 
    "data/aggregated/hexagon/demand_hex_6h_medium.parquet",
    "data/aggregated/hexagon/demand_hex_24h_low.parquet", 
    "data/aggregated/hexagon/demand_hex_24h_medium.parquet"
]

# Strip out coordinates AND concurrent trip aggregates to eliminate target leakage
LEAKING_COLS = [
    'pickup_centroid_latitude', 'pickup_centroid_longitude',
    'dropoff_centroid_latitude', 'dropoff_centroid_longitude',
    'trip_seconds', 'trip_miles', 'fare', 'tips', 'tolls', 'extras', 'trip_total',
    'active_taxis', 'avg_idle_time' # active_taxis also heavily leaks concurrent demand
]

# Explicit list of valid spatial identifiers
SPATIAL_OPTIONS = ["pickup_h3_res6", "pickup_h3_res7", "pickup_community_area"]

results = []

for file in files:
    print(f"Processing {file}...")
    try:
        df = pd.read_parquet(file)
        zero_demand_pct = (df['trip_count'] == 0).mean() * 100
        df = df.sort_values(by='bucket_index')
        
        # 1. Dynamically select the exact spatial column based on its name
        spatial_col = next((col for col in SPATIAL_OPTIONS if col in df.columns), None)

        if not spatial_col:
            print(f"  -> Skipping {file}: No matching spatial column found.")
            continue

        # 2. Separate Target (y) and Features (X)
        y = df['trip_count']
        
        # Drop target, coordinates, and unneeded identifiers from X
        cols_to_drop = ['trip_count', 'time_bucket', 'bucket_index'] + LEAKING_COLS
        X = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')
        
        # 3. Identify Numeric vs. Spatial Categorical columns for scaling/encoding
        numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        if spatial_col in numeric_cols:
            numeric_cols.remove(spatial_col) # Treat as a category for One-Hot encoding, not a continuous scale
            
        categorical_cols = [spatial_col]

        # 4. Chronological Train/Test Split (80/20)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
        
        # 5. Build Preprocessor pipeline
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', StandardScaler(), numeric_cols),
                ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_cols)
            ]
        )
        
        X_train_processed = preprocessor.fit_transform(X_train)
        X_test_processed = preprocessor.transform(X_test)
        
        # 6. Train the Baseline LinearSVR
        model = LinearSVR(loss='squared_epsilon_insensitive', dual=False, random_state=42)
        model.fit(X_train_processed, y_train)
        
        # 7. Evaluate
        y_pred = model.predict(X_test_processed)
        y_pred = np.maximum(y_pred, 0) # Clip negative predictions
        
        r2 = r2_score(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mean_y = y_test.mean()
        nrmse = (rmse / (mean_y + 1e-9)) * 100 
        
        results.append({
            "Dataset": file,
            "Spatial Feature Used": spatial_col,
            "Rows": len(df),
            "Zero Demand (%)": round(zero_demand_pct, 2),
            "R-Squared": round(r2, 4),
            "NRMSE (%)": round(nrmse, 2)
        })
        
    except FileNotFoundError:
        print(f"  -> Error: File {file} not found. Skipping.")
    except Exception as e:
        print(f"  -> Error processing {file}: {e}")

# Display Final Results Table
results_df = pd.DataFrame(results)
print("\n=== PURE SPATIO-TEMPORAL RESOLUTION COMPARISON ===")
print(results_df.to_string(index=False))

Processing data/aggregated/community_area/demand_ca_1h.parquet...
Processing data/aggregated/community_area/demand_ca_2h.parquet...
Processing data/aggregated/community_area/demand_ca_6h.parquet...
Processing data/aggregated/community_area/demand_ca_24h.parquet...
Processing data/aggregated/hexagon/demand_hex_1h_low.parquet...
Processing data/aggregated/hexagon/demand_hex_1h_medium.parquet...
Processing data/aggregated/hexagon/demand_hex_2h_low.parquet...
Processing data/aggregated/hexagon/demand_hex_2h_medium.parquet...
Processing data/aggregated/hexagon/demand_hex_6h_low.parquet...
Processing data/aggregated/hexagon/demand_hex_6h_medium.parquet...
Processing data/aggregated/hexagon/demand_hex_24h_low.parquet...
Processing data/aggregated/hexagon/demand_hex_24h_medium.parquet...

=== PURE SPATIO-TEMPORAL RESOLUTION COMPARISON ===
                                              Dataset  Spatial Feature Used    Rows  Zero Demand (%)  R-Squared  NRMSE (%)
  data/aggregated/community_area/d

## Spatio-Temporal Resolution Analysis & Downstream Strategy

An initial analysis of the baseline model performance reveals a striking mathematical trend: **as data is aggregated over longer time horizons ($1\text{h} \rightarrow 2\text{h} \rightarrow 6\text{h} \rightarrow 24\text{h}$), the underlying dataset distribution radically smooths out.** Zero-demand periods drop precipitously, variance stabilizes, and the baseline $R^2$ performance naturally climbs from the high-0.50s to nearly 0.90.

However, selecting the right modeling targets requires balancing raw statistical performance with real-world operational utility. Predictive models tracking a 24-hour average are helpful for macro planning but useless for a live fleet coordinator managing real-time driver shifts. 

By filtering down the initial options, we have selected five distinct datasets that form a progressive, highly strategic pipeline.

---

### Chosen Modeling Candidates

| Dataset | Spatial Grid | Temporal Window | Rows | Zero Demand | Baseline $R^2$ | Strategic Testing Profile |
| :--- | :---: | :---: | :---: | :---: | :---: | :--- |
| `demand_hex_1h_low.parquet` | Hexagon (Res 6) | 1 Hour | 289,080 | 31.52% | 0.5971 | **The High-Frequency Operational Frontier** |
| `demand_hex_2h_low.parquet` | Hexagon (Res 6) | 2 Hours | 144,540 | 23.73% | 0.6089 | **The High-Definition Near-Real-Time Window** |
| `demand_hex_6h_low.parquet` | Hexagon (Res 6) | 6 Hours | 48,180 | 14.44% | 0.6606 | **The Shift-Level Operational Sweet Spot** |
| `demand_hex_24h_medium.parquet`| Hexagon (Res 7) | 24 Hours | 58,400 | 7.00% | 0.8881 | **The Strategic Macro Champion** |
| `demand_ca_24h.parquet` | Community Area | 24 Hours | 28,105 | 1.13% | 0.8807 | **The Administrative Grid Showdown** |

---

### Architectural Rationale: Why These 5 Models?

### 1. The Real-Time Benchmark: Hexagon 1h Low (`demand_hex_1h_low`)
* **The Challenge:** At 1 hour, this dataset is highly chaotic, noisy, and heavily impacted by a 31.52% zero-demand wall. 
* **Why we keep it:** This is the baseline for live vehicle dispatching. Keeping this configuration allows us to test feature engineering and see if clean temporal cyclical indicators (e.g., sine/cosine transformations) can inject structural order into short-term taxi demand patterns.

### 2. The Scaled Frontier: Hexagon 2h Low (`demand_hex_2h_low`)
* **The Challenge:** At 144,540 rows, it is too large for an exact non-linear SVM kernel grid search to execute on consumer hardware without memory exhaustion.
* **Why we keep it:** It serves as the ideal testing ground for **Nystroëm Kernel Approximation**. By slightly widening the window from 1h to 2h, we reduce some noise while maintaining strict, actionable real-time dispatch intervals. This target evaluates whether kernel approximation can push a standard linear baseline into high-tier performance.

### 3. The Computing Sweet Spot: Hexagon 6h Low (`demand_hex_6h_low`)
* **The Challenge:** Finding an operational balance where data is clean but still highly actionable for fleet logistics.
* **Why we keep it:** At ~48,000 rows and only 14.44% zero inflation, this dataset is small enough that our machine can run an **exact, unapproximated non-linear RBF kernel configuration** without risking memory page faults or system crashes. It maps perfectly to standard driver shifts (Morning Rush, Afternoon, Evening, Night).

### 4. Exploiting the Loophole: Hexagon 24h Medium (`demand_hex_24h_medium`)
* **The Challenge:** Finer spatial resolutions (Resolution 7 / Medium) usually cause zero-demand rows to skyrocket because smaller geographic pixels are harder to fill consistently. 
* **Why we keep it:** The 24-hour temporal aggregation window completely absorbs this penalty. Zero inflation drops to an all-time low of **7.00%**, matching the lower spatial resolution model (`24h_low` at 7.04%) but offering **7x sharper geographic detail**. This allows an SVM to focus completely on intricate neighborhood-level infrastructure differences rather than fighting a sparse target array.

### 5. The Shape Showdown: Community Area 24h (`demand_ca_24h`)
* **The Challenge:** Determining whether machine learning algorithms perform better on uniform mathematical structures or historical human boundaries.
* **Why we keep it:** With a negligible 1.13% zero-demand rate, the target sparsity problem is completely eliminated. Pitting this directly against `demand_hex_24h_medium` creates an elegant macro experiment: it reveals whether distance-based models adapt better to **rigid, equidistant geographic pixels (hexagons)** or **irregular, human-defined administrative zones**.

---

### Datasets Dropped from the Pipeline

* **Hexagon 1h & 2h Medium:** Dropped entirely. At 700k and 1.4M rows mixed with severe zero inflation (up to 54.72%), these datasets represent a computational worst-case scenario for an SVM framework without providing any structural advantages over their cleaner, low-resolution counterparts.
* **Hexagon 24h Low:** Dropped in favor of `24h_medium`. Because the 24-hour window eliminated the spatial sparsity penalty, choosing the medium resolution allows us to preserve sharp neighborhood definitions for macro-capacity planning at no statistical cost.

## Finalized Spatio-Temporal Strategy: The 4 Strategic Candidates

By analyzing the data size, sparsity patterns, and baseline efficiencies across all Parquet files, we have finalized a 4-candidate strategic roadmap. This selection balances near-real-time operational utility with high-accuracy macro planning, while introducing a direct architectural showdown between uniform hexagon cells and irregular administrative boundaries.

---

### Master SVM Candidate Matrix

| Dataset | Strategic Profile | Rows | Zero Demand | Baseline R² | Core Machine Learning Objective |
| :--- | :--- | :---: | :---: | :---: | :--- |
| `demand_hex_6h_low.parquet` | **Operational Sweet Spot** | 48,180 | 14.44% | 0.6606 | Train an **exact, unapproximated RBF kernel** via Grid Search without computational bottlenecks. |
| `demand_hex_2h_low.parquet` | **High-Definition Frontier** | 144,540 | 23.73% | 0.6089 | Deploy the scaled **Nystroëm pipeline** to capture sharp, near-real-time dispatch shift windows. |
| `demand_hex_24h_medium.parquet`| **Hexagonal Macro-Champion** | 58,400 | 7.00% | 0.8881 | Exploit the 24h window to bypass the resolution penalty, giving the SVM 7x sharper spatial resolution. |
| `demand_ca_24h.parquet` | **Administrative Macro-Baseline**| 28,105 | 1.13% | 0.8807 | Evaluate SVM performance on **irregular social/political shapes** with near-zero target sparsity. |

---

### Detailed Rationale for the Add Silicon Candidate: Community Area 24h

The inclusion of `demand_ca_24h.parquet` serves a vital diagnostic purpose in our SVM framework:

* **Elimination of Target Sparsity:** With a negligible 1.13% zero-demand rate, the mathematical "brick wall" at zero disappears. The SVM regressor is freed from the constraint of handling compressed zero-inflation boundaries, enabling a pure test of variance capture.
* **The Shape Showdown:** Community areas represent human-defined, political, and socio-economic neighborhoods which vary drastically in size and perimeter. Hexagons represent rigid, equidistant geographic pixels. Testing this dataset alongside `demand_hex_24h_medium` will reveal if distance-based kernels (RBF) generalize better on mathematical grids or historical human-density boundaries.
* **Maximum Computational Efficiency:** At under 30,000 total rows, this dataset allows for exhaustive hyperparameter tuning sweeps ($C$, $\gamma$, $\epsilon$) to be completed on an M2 chip in seconds, providing a pristine, unapproximated peak-performance anchor.